To deeply understand Message Passing in GNN

In [3]:
import numpy as np

# Basic Message Passing Process
The general message passing process consists of three steps:

+ Message Creation: Each node creates messages for its neighbours
  + Message function: transform node features into messages
  + $ m_{ij}^{(l)} = Message(h_i^{(l)}, h_j^{(l)}, e_{ij}) $  <-- Message from $j$ to $i$
    + simple linear
    + 

+ Message Aggregation: Nodes collect and combine messages from neighbors
  + Aggregate function:
  + $ m_i^{(l)} = Aggregate({m_{ij}^{(l)} : j ∈ N(i)}) $ <-- Aggregate messages
    + Sum: torch.sum(messages, dim=neighbor_dim)
    + Mean: torch.mean(messages, dim=neighbor_dim)
    + Max: torch.max(messages, dim=neighbor_dim)
    + Attention: Weighted combination based on learned attention weights

+ Node Update: Nodes update their representations using aggregated messages
  + Update function:
  + $ h_i^{(l+1)} = Update(h_i^{(l)}, m_i^{(l)})$  <-- Update node representation

# Example exercise
- Nodes: 5 people (Alice, Bob, Charlie, Eve, Dave)
- Features: Age, income, education level (normalised)
- Edges: Friendship connections
- Task: Learn representations that capture social influence

=== Key Takeaways ===
1. Message Passing allows nodes to share information with neighbors
2. Multiple layers enable information to travel further in the graph
3. Node representations become influenced by their local neighborhood
4. The process is permutation-invariant (order of neighbors doesn't matter)
5. Aggregation functions (sum, mean, max) determine how messages combine

In [13]:
# Features: [age, income, education] (normalized)
features = np.array([
    [0.3, 0.6, 0.7],  # Alice
    [0.4, 0.5, 0.6],  # Bob
    [0.6, 0.4, 0.5],  # Charlie
    [0.2, 0.7, 0.9],  # Eve
    [0.5, 0.3, 0.6],  # Dav
])

# Adjacency matrix for undirected graph
adjacency = np.array([
    [0, 1, 0, 1, 0],  # Alice
    [1, 0, 1, 0, 0],  # Bob
    [0, 1, 0, 1, 1],  # Charlie
    [1, 0, 1, 0, 1],  # Eve
    [0, 0, 1, 1, 0],  # Dave
])

# Weights for linear transformation
weights = np.array([
    [0.2, 0.4],
    [0.3, 0.1],
    [0.5, 0.3]
])

convert to code using numpy

In [9]:
def sigmoid(x):
    """Sigmoid activation function"""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def relu(x):
    """ReLU activation function"""
    return np.maximum(0, x)

class SimpleGNN:
    """
    A simple Graph Neural Network implementation using only NumPy
    """
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Initialize weights randomly
        np.random.seed(42)  # For reproducible results
        
        # Message network weights
        self.W_message = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b_message = np.zeros(hidden_dim)
        
        # Update network weights
        self.W_update = np.random.randn(input_dim + hidden_dim, output_dim) * 0.1
        self.b_update = np.zeros(output_dim)
    
    def create_messages(self, node_features):
        """
        Step 1: Create messages for each node
        """
        messages = np.dot(node_features, self.W_message) + self.b_message
        messages = relu(messages)  # Apply activation
        return messages
    
    def aggregate_messages(self, messages, adjacency_matrix):
        """
        Step 2: Aggregate messages from neighbors
        """
        # Sum aggregation: each node receives sum of neighbor messages
        aggregated = np.dot(adjacency_matrix, messages)
        return aggregated
    
    def update_nodes(self, node_features, aggregated_messages):
        """
        Step 3: Update node representations
        """
        # Concatenate original features with aggregated messages
        combined = np.concatenate([node_features, aggregated_messages], axis=1)
        
        # Apply linear transformation
        updated = np.dot(combined, self.W_update) + self.b_update
        updated = sigmoid(updated)  # Apply activation
        
        return updated
    
    def forward(self, node_features, adjacency_matrix):
        """
        Complete forward pass
        """
        # Step 1: Create messages
        messages = self.create_messages(node_features)
        
        # Step 2: Aggregate messages
        aggregated = self.aggregate_messages(messages, adjacency_matrix)
        
        # Step 3: Update nodes
        updated_features = self.update_nodes(node_features, aggregated)
        
        return updated_features, messages, aggregated

def create_social_network_dataset():
    """
    Create a small social network dataset
    
    Graph: Friend network with 5 people
    Node features: [age, income, education_level]
    
    Network structure:
        Alice (0) -- Bob (1) -- Charlie (2)
          |                      |
        Dave (4) ----------- Eve (3)
    """
    
    # Node features: [age, income_k, education_years]
    node_features = np.array([
        [25, 50, 16],   # Alice: 25 years old, 50k income, 16 years education
        [30, 75, 18],   # Bob: 30 years old, 75k income, 18 years education  
        [35, 60, 14],   # Charlie: 35 years old, 60k income, 14 years education
        [28, 80, 20],   # Eve: 28 years old, 80k income, 20 years education
        [32, 45, 12],   # Dave: 32 years old, 45k income, 12 years education
    ], dtype=np.float32)
    
    # Normalize features to [0, 1] range
    node_features = (node_features - node_features.min(axis=0)) / (node_features.max(axis=0) - node_features.min(axis=0))
    
    # Adjacency matrix (undirected friendships)
    adjacency_matrix = np.array([
        [0, 1, 0, 0, 1],  # Alice friends with Bob, Dave
        [1, 0, 1, 0, 0],  # Bob friends with Alice, Charlie
        [0, 1, 0, 1, 0],  # Charlie friends with Bob, Eve
        [0, 0, 1, 0, 1],  # Eve friends with Charlie, Dave
        [1, 0, 0, 1, 0],  # Dave friends with Alice, Eve
    ], dtype=np.float32)
    
    names = ["Alice", "Bob", "Charlie", "Eve", "Dave"]
    
    return node_features, adjacency_matrix, names

def demonstrate_step_by_step():
    """
    Demonstrate message passing step by step
    """
    print("=== Message Passing GNN Demo with Social Network ===\n")
    
    # Create dataset
    node_features, adjacency_matrix, names = create_social_network_dataset()
    
    print("Social Network Dataset:")
    print("People: Alice, Bob, Charlie, Eve, Dave")
    print("Features: [age, income, education_level] (normalized)")
    print("\nInitial node features:")
    for i, name in enumerate(names):
        print(f"{name:8}: {node_features[i]}")
    
    print("\nFriendship network (adjacency matrix):")
    print("    ", "  ".join(f"{name:>5}" for name in names))
    for i, name in enumerate(names):
        print(f"{name:8}", adjacency_matrix[i])
    
    # Create GNN
    gnn = SimpleGNN(input_dim=3, hidden_dim=4, output_dim=2)
    
    print("\n" + "="*50)
    print("MESSAGE PASSING PROCESS")
    print("="*50)
    
    # Forward pass with detailed steps
    updated_features, messages, aggregated = gnn.forward(node_features, adjacency_matrix)
    
    print("\nStep 1: Message Creation")
    print("Each person creates a message based on their features:")
    for i, name in enumerate(names):
        print(f"{name:8}: {messages[i]}")
    
    print("\nStep 2: Message Aggregation") 
    print("Each person collects messages from their friends:")
    for i, name in enumerate(names):
        friends = [names[j] for j in range(len(names)) if adjacency_matrix[i][j] == 1]
        print(f"{name:8}: friends with {friends}")
        print(f"         aggregated message: {aggregated[i]}")
    
    print("\nStep 3: Node Update")
    print("Each person updates their representation:")
    for i, name in enumerate(names):
        print(f"{name:8}: original {node_features[i]} -> updated {updated_features[i]}")
    
    return updated_features

def demonstrate_multi_layer():
    """
    Demonstrate multiple layers of message passing
    """
    print("\n\n=== Multi-Layer Message Passing ===\n")
    
    node_features, adjacency_matrix, names = create_social_network_dataset()
    
    # Create multiple GNN layers
    layer1 = SimpleGNN(input_dim=3, hidden_dim=4, output_dim=3)
    layer2 = SimpleGNN(input_dim=3, hidden_dim=4, output_dim=2)
    
    print("Initial features:")
    for i, name in enumerate(names):
        print(f"{name:8}: {node_features[i]}")
    
    # Layer 1
    features_l1, _, _ = layer1.forward(node_features, adjacency_matrix)
    print("\nAfter Layer 1 (1-hop neighborhood information):")
    for i, name in enumerate(names):
        print(f"{name:8}: {features_l1[i]}")
    
    # Layer 2  
    features_l2, _, _ = layer2.forward(features_l1, adjacency_matrix)
    print("\nAfter Layer 2 (2-hop neighborhood information):")
    for i, name in enumerate(names):
        print(f"{name:8}: {features_l2[i]}")
    
    print("\nInformation propagation:")
    print("- Layer 1: Each person learns from direct friends")
    print("- Layer 2: Each person learns from friends of friends")
    
    return features_l2

def analyze_information_flow():
    """
    Analyze how information flows through the network
    """
    print("\n\n=== Information Flow Analysis ===\n")
    
    node_features, adjacency_matrix, names = create_social_network_dataset()
    
    # Create a simple scenario: only Alice has high education initially
    scenario_features = np.zeros_like(node_features)
    scenario_features[0] = [1.0, 0.0, 1.0]  # Alice has high education
    
    print("Scenario: Only Alice starts with high education feature")
    print("Let's see how this information spreads:")
    
    # display input features
    print(f"\nInput features:")
    for i, name in enumerate(names):
        education_influence = scenario_features[i]
        print(f"{name:8}: education influence = {education_influence}")
    
    gnn = SimpleGNN(input_dim=3, hidden_dim=4, output_dim=3)
    
    current_features = scenario_features.copy()
    
    for layer in range(3):
        print(f"\nLayer {layer + 1}:")
        updated_features, _, _ = gnn.forward(current_features, adjacency_matrix)
        
        for i, name in enumerate(names):
            education_influence = updated_features[i]
            print(f"{name:8}: education influence = {education_influence}")
        
        #current_features = np.column_stack([updated_features, current_features[:, 1:]])
        current_features = updated_features
    print("\nObservation: Information spreads through the network")
    print("- Direct friends receive information first")
    print("- Information reaches distant nodes in later layers")

In [4]:
# Basic message passing
demonstrate_step_by_step()

=== Message Passing GNN Demo with Social Network ===

Social Network Dataset:
People: Alice, Bob, Charlie, Eve, Dave
Features: [age, income, education_level] (normalized)

Initial node features:
Alice   : [0.         0.14285715 0.5       ]
Bob     : [0.5        0.85714287 0.75      ]
Charlie : [1.         0.42857143 0.25      ]
Eve     : [0.3 1.  1. ]
Dave    : [0.7 0.  0. ]

Friendship network (adjacency matrix):
     Alice    Bob  Charlie    Eve   Dave
Alice    [0. 1. 0. 0. 1.]
Bob      [1. 0. 1. 0. 0.]
Charlie  [0. 1. 0. 1. 0.]
Eve      [0. 0. 1. 0. 1.]
Dave     [1. 0. 0. 1. 0.]

MESSAGE PASSING PROCESS

Step 1: Message Creation
Each person creates a message based on their features:
Alice   : [0.         0.02378319 0.         0.        ]
Bob     : [0.         0.01370991 0.1329892  0.10700188]
Charlie : [0.02789941 0.         0.12086396 0.1735498 ]
Eve     : [0.         0.02669438 0.13101017 0.0758614 ]
Dave    : [0.03476999 0.         0.0453382  0.10661209]

Step 2: Message Aggregat

array([[0.47829974, 0.49485565],
       [0.44537059, 0.46504119],
       [0.48068467, 0.43928435],
       [0.42890117, 0.47212263],
       [0.50527244, 0.46185092]])

In [5]:
# Multi-layer message passing
demonstrate_multi_layer()



=== Multi-Layer Message Passing ===

Initial features:
Alice   : [0.         0.14285715 0.5       ]
Bob     : [0.5        0.85714287 0.75      ]
Charlie : [1.         0.42857143 0.25      ]
Eve     : [0.3 1.  1. ]
Dave    : [0.7 0.  0. ]

After Layer 1 (1-hop neighborhood information):
Alice   : [0.48472505 0.48604335 0.51643154]
Bob     : [0.47202271 0.43461115 0.51003392]
Charlie : [0.49352872 0.43743466 0.46630593]
Eve     : [0.46212257 0.43595926 0.52799248]
Dave    : [0.50363665 0.46824603 0.46742668]

After Layer 2 (2-hop neighborhood information):
Alice   : [0.46754127 0.4687384 ]
Bob     : [0.46980868 0.46990714]
Charlie : [0.47125533 0.46892515]
Eve     : [0.46909636 0.47050095]
Dave    : [0.46986502 0.46773541]

Information propagation:
- Layer 1: Each person learns from direct friends
- Layer 2: Each person learns from friends of friends


array([[0.46754127, 0.4687384 ],
       [0.46980868, 0.46990714],
       [0.47125533, 0.46892515],
       [0.46909636, 0.47050095],
       [0.46986502, 0.46773541]])

In [10]:
# Information flow analysis
analyze_information_flow()



=== Information Flow Analysis ===

Scenario: Only Alice starts with high education feature
Let's see how this information spreads:

Input features:
Alice   : education influence = [1. 0. 1.]
Bob     : education influence = [0. 0. 0.]
Charlie : education influence = [0. 0. 0.]
Eve     : education influence = [0. 0. 0.]
Dave    : education influence = [0. 0. 0.]

Layer 1:
Alice   : education influence = [0.48335461 0.41761826 0.49351864]
Bob     : education influence = [0.49801702 0.50473591 0.49856957]
Charlie : education influence = [0.5 0.5 0.5]
Eve     : education influence = [0.5 0.5 0.5]
Dave    : education influence = [0.49801702 0.50473591 0.49856957]

Layer 2:
Alice   : education influence = [0.48454834 0.45484349 0.49868233]
Bob     : education influence = [0.48326027 0.45161197 0.49897177]
Charlie : education influence = [0.48333839 0.45176441 0.49884963]
Eve     : education influence = [0.48333839 0.45176441 0.49884963]
Dave    : education influence = [0.48326027 0.45161197

In [14]:
adjacency

array([[0, 1, 0, 1, 0],
       [1, 0, 1, 0, 0],
       [0, 1, 0, 1, 1],
       [1, 0, 1, 0, 1],
       [0, 0, 1, 1, 0]])